# Unified Lab 2: The Evolutionary Engineer

## The Scenario
You are an Optimization Engineer at **EvoTech**, an AI firm that borrows mechanics from Darwinian evolution to solve NP-Hard configuration problems. Your current contract is with the Global Defense Initiative. They need to deploy $N$ autonomous defense drones on an $N \times N$ grid.

The constraint is absolute: **No two drones can share the same row, column, or diagonal line of sight.** Because an $8 \times 8$ grid has over 4 billion possible configurations, and a $16 \times 16$ grid is mathematically impossible to brute-force, Uninformed Search (like BFS) will fail. Instead, you will build a Genetic Algorithm from scratch. You will encode drone positions as digital "DNA," and allow the best configurations to crossbreed and mutate until a perfect, conflict-free defense grid evolves.

## Milestone 1: Set Up

Run this cell first.

In [ ]:
# MILESTONE 1 SETUP - RUN THIS CELL FIRST
import random
import numpy as np
import matplotlib.pyplot as plt

N_DRONES = 8

def generate_random_chromosome(n):
    """
    Generates a random initial configuration.
    Returns a list of length n, with values ranging from 0 to n-1.
    """
    return [random.randint(0, n - 1) for _ in range(n)]

# Generate two random parents
parent_A = generate_random_chromosome(N_DRONES)
parent_B = generate_random_chromosome(N_DRONES)

print(f"EvoTech Genetic Engine Initialized for a {N_DRONES}x{N_DRONES} grid.")
print(f"Parent A DNA: {parent_A}")
print(f"Parent B DNA: {parent_B}")

## Milestone 1: The Gene Splicer (Crossover & Mutation)
**Objective:** Before we can simulate a full evolutionary lifecycle, we need to build the genetic operators that make reproduction possible.

You will use an **Integer Encoding** scheme. A "chromosome" is a list of $N$ integers. The *index* of the list represents the row, and the *value* represents the column. For example, a chromosome `[2, 0, 3, 1]` means drones are placed at `(row 0, col 2)`, `(row 1, col 0)`, `(row 2, col 3)`, and `(row 3, col 1)`. Notice that this encoding automatically prevents two drones from ever sharing the same row!

Your task is to write the functions for **Single-Point Crossover** and **Mutation**.

### The Architect's Blueprint
*Write the `single_point_crossover(parent1, parent2)` and `mutate(chromosome, mutation_rate)` functions.*

* **1. Single-Point Crossover:** * Pick a random integer `crossover_point` between `1` and `len(parent1) - 1`. (Hint: use `random.randint`).
    * Create `child1` by slicing `parent1` from the start up to the crossover point, and combining it with a slice of `parent2` from the crossover point to the end.
    * Create `child2` by doing the exact opposite (first part of `parent2`, second part of `parent1`).
    * Return `child1, child2`.
* **2. Mutation:** * Iterate through the `chromosome` list.
    * For each gene (column position), generate a random float between `0.0` and `1.0` using `random.random()`.
    * If that random number is *less than* the `mutation_rate`, replace that gene with a brand new random integer between `0` and `N_DRONES - 1`.
    * Return the mutated chromosome.

In [ ]:
def single_point_crossover(parent1, parent2):
    crossover_point = random.randint(1, len(parent1) - 1)
    child1 = parent1[:crossover_point] + parent2[crossover_point:]
    child2 = parent2[:crossover_point] + parent1[crossover_point:]
    return child1, child2


def mutate(chromosome, mutation_rate):
    for i in range(len(chromosome)):
        if random.random() < mutation_rate:
            chromosome[i] = random.randint(0, N_DRONES - 1)
    return chromosome

### The Test Harness
*Run this code to test if your genetic operators are splicing and mutating DNA correctly.*

In [ ]:
# MILESTONE 1 TEST HARNESS

# Test Crossover
child_A, child_B = single_point_crossover(parent_A, parent_B)

print(f"Parent A: {parent_A}")
print(f"Parent B: {parent_B}")
print(f"Child A:  {child_A}")
print(f"Child B:  {child_B}")

assert len(child_A) == N_DRONES and len(child_B) == N_DRONES, "Failure: Crossover altered the length of the chromosome!"
assert child_A != parent_A and child_B != parent_B, "Failure: Crossover did not mix the DNA. Children are identical to parents."

# Test Mutation
# We use a 100% mutation rate just to verify the function works
mutated_child = mutate(child_A.copy(), mutation_rate=1.0)
print(f"\nChild A (Before 100% Mutation): {child_A}")
print(f"Child A (After 100% Mutation):  {mutated_child}")

assert mutated_child != child_A, "Failure: Mutation function failed to alter the genes."

print("\nSUCCESS! Genetic operators are functioning within acceptable parameters.")

### Architect's Audit (Milestone 1)

1. **Crossover Dynamics:** You implemented *Single-Point Crossover*. Based on Lecture 2.3, what is *Uniform Crossover*? If you were trying to preserve highly specific patterns of drones scattered all across the board, why might Uniform Crossover destroy those patterns faster than Single-Point?
2. **The Mutation Dial:** Based on Lecture 2.4, explain the role of mutation in a Genetic Algorithm. What happens to the AI's search if you set the mutation rate to `0.0`? What happens if you set it to `0.99`?

ENTER YOUR ANSWERS HERE

## Milestone 2: Set Up

Run this cell first

In [ ]:
# MILESTONE 2 SETUP - RUN THIS CELL FIRST
import random
from collections import Counter

def calculate_fitness(chromosome):
    """
    Calculates the fitness of a chromosome.
    Max fitness for N=8 is 28 (all pairs non-in-sight).
    """
    n = len(chromosome)
    max_fitness = (n * (n - 1)) // 2
    conflicts = 0

    for i in range(n):
        for j in range(i + 1, n):
            # Check for column conflict
            if chromosome[i] == chromosome[j]:
                conflicts += 1
            # Check for diagonal conflict
            elif abs(chromosome[i] - chromosome[j]) == abs(i - j):
                conflicts += 1

    return max_fitness - conflicts

def print_board(chromosome):
    """Visually prints the NxN drone grid."""
    n = len(chromosome)
    print("\n+" + "---+" * n)
    for row in range(n):
        row_str = "|"
        for col in range(n):
            if chromosome[row] == col:
                row_str += " \033[91mD\033[0m |" # Red D for Drone
            else:
                row_str += "   |"
        print(row_str)
        print("+" + "---+" * n)

print("Fitness Engine successfully loaded.")

## Milestone 2: Natural Selection (Roulette vs. Tournament)
**Objective:** In nature, survival of the fittest dictates which individuals get to pass their genes to the next generation. In a Genetic Algorithm, we use a **Fitness Function** to map a chromosome to a numerical score (a "Fitness Landscape"). We then use **Selection Strategies** to probabilistically choose parents based on those scores.

To solve our drone defense problem, the objective is to maximize the number of non-attacking drone pairs. For an $8 \times 8$ grid, there are 28 possible pairs of drones. Therefore, a perfect, conflict-free board has a maximum fitness score of exactly **28**.

You will implement two distinct selection strategies covered in Lecture 2.2: **Roulette Wheel Selection** and **Tournament Selection**.

### The Architect's Blueprint
*Write the `roulette_wheel_selection(population, fitness_scores)` and `tournament_selection(population, fitness_scores, k)` functions.*

* **1. Roulette Wheel Selection:**
    * *Logic:* The probability of an individual being selected is directly proportional to its fitness score.
    * *Task:* Calculate the `total_fitness` of the entire population. Generate a random integer/float between 0 and `total_fitness`. Iterate through the population, keeping a running sum of their fitness scores. The moment your running sum meets or exceeds your random number, return that specific chromosome.
* **2. Tournament Selection:**
    * *Logic:* Randomly grab a subset of the population. They fight. The one with the highest fitness in that small group wins the right to reproduce.
    * *Task:* Randomly select `k` indices from the population. Find the individual among those `k` contenders that has the highest corresponding fitness score. Return that chromosome.

In [ ]:
# ENTER YOUR CODE HERE

### The Test Harness
*Run this code to test your selection functions. We will simulate picking parents 1,000 times to see how "Selection Pressure" affects the gene pool!*

In [ ]:
# MILESTONE 2 TEST HARNESS

# 1. Create a tiny test population of 4 drones with close fitness scores
test_population = [
    [0, 1, 2, 3, 4, 5, 6, 7], # Terrible (Diagonal conflict on every drone)
    [0, 0, 2, 2, 4, 4, 6, 6], # Poor (Many column and diagonal conflicts)
    [0, 2, 4, 6, 1, 2, 3, 7], # Good (Only a few diagonal conflicts)
    [0, 6, 4, 7, 1, 3, 5, 2]  # Perfect (Fitness = 28)
]
test_fitness = [calculate_fitness(chrom) for chrom in test_population]

print(f"Population Fitness Scores: {test_fitness}")

# 2. Test Roulette Wheel Selection
roulette_winners = []
for _ in range(1000):
    winner = roulette_wheel_selection(test_population, test_fitness)
    roulette_winners.append(test_population.index(winner))

# 3. Test Tournament Selection (k=2)
tournament_winners = []
for _ in range(1000):
    winner = tournament_selection(test_population, test_fitness, k=2)
    tournament_winners.append(test_population.index(winner))

print("\n--- Selection Frequency (Out of 1000 picks) ---")
print(f"Roulette Wheel: {Counter(roulette_winners)}")
print(f"Tournament (k=2): {Counter(tournament_winners)}")

assert sum(Counter(roulette_winners).values()) == 1000, "Failure: Roulette selection dropped individuals."
assert sum(Counter(tournament_winners).values()) == 1000, "Failure: Tournament selection dropped individuals."
print("\nSUCCESS! Selection functions are ready for the GA Loop.")

In [ ]:
for c in test_population:
  print_board(c)

### Architect's Audit (Milestone 2)

1. **Selection Pressure:** Look at your Test Harness printout. Notice how the frequencies differ between Roulette and Tournament. Define "Selection Pressure." Which method exerted higher selection pressure toward the absolute best individual?
2. **The Flaw in the Wheel:** According to Lecture 2.2, Roulette Wheel selection suffers when the population's fitness values get very close to each other (e.g., late in the evolutionary cycle when everyone is highly fit). Why does this happen mathematically, and how does Tournament Selection naturally fix this problem without needing to scale the fitness values?

ENTER YOUR ANSWERS HERE

## Milestone 3: Set Up

Run this cell first.

In [ ]:
# MILESTONE 3 SETUP - RUN THIS CELL FIRST
import matplotlib.pyplot as plt

def get_max_fitness(n):
    """
    Calculates the maximum possible fitness for an N-Drones board.
    Fitness is based on the total number of non-attacking pairs.

    Parameters:
    n (int): The size of the board (N x N) and number of drones.

    Returns:
    int: The maximum fitness score (C(n, 2)).
    """
    # Using integer division (//) to return a clean integer
    return (n * (n - 1)) // 2

def plot_convergence(best_fitness_history, avg_fitness_history, n_drones):
    """Plots the fitness metrics over generations to visualize convergence."""
    plt.figure(figsize=(10, 6))
    plt.plot(best_fitness_history, label='Best Fitness', color='#2ECC40', linewidth=2)
    plt.plot(avg_fitness_history, label='Average Fitness', color='#0074D9', linestyle='--')
    max_fitness = get_max_fitness(n_drones)
    plt.axhline(y=max_fitness, color='#FF4136', linestyle=':', label=f'Global Optimum ({max_fitness})')

    plt.title('Genetic Algorithm Convergence Dynamics', fontsize=14, fontweight='bold')
    plt.xlabel('Generation', fontsize=12)
    plt.ylabel('Fitness Score', fontsize=12)
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.show()

print("Graphing Engine successfully loaded.")

## Milestone 3: The GA Lifecycle
**Objective:** It is time to solve the problem. You have built the DNA (Integer Encoding), the reproductive operators (Crossover & Mutation), and the survival mechanics (Fitness & Selection). Now, you must assemble them into the main Genetic Algorithm lifecycle loop.

According to Lecture 2.4, the GA Lifecycle is: **Initialize $\rightarrow$ Evaluate $\rightarrow$ Select $\rightarrow$ Crossover $\rightarrow$ Mutate $\rightarrow$ Replace.**

You will write the main engine that loops through these phases over multiple generations. You will also track the "Best Fitness" and "Average Fitness" of your population over time so we can graph your algorithm's **Convergence Dynamics**.

### The Architect's Blueprint
*Write the `run_genetic_algorithm(n_drones, pop_size, max_generations, mutation_rate)` function.*

* **1. Initialization:** Create an initial `population` list containing `pop_size` random chromosomes. (Use your `generate_random_chromosome()` function).
* **2. The Lifecycle Loop:** Create a `for` loop that runs `max_generations` times. Inside the loop:
    * **Evaluate:** Calculate the fitness for every chromosome in the current population.
    * **Track Metrics:** Find the highest fitness in the population and append it to a `best_history` list. Calculate the average fitness and append it to an `avg_history` list.
    * **Check for Win:** If the best fitness is `28` (a perfect, conflict-free board), `break` out of the loop early! You solved it!
    * **Create Next Generation:** Initialize an empty `new_population` list.
    * **Reproduce:** Use a `while` loop that runs until `new_population` is the same size as `pop_size`.
        * *Select:* Pick `parent1` and `parent2` using `tournament_selection` (or roulette, if you prefer).
        * *Crossover:* Pass the parents into `single_point_crossover` to get `child1` and `child2`.
        * *Mutate:* Pass both children into `mutate` using the `mutation_rate`.
        * *Replace:* Append the children to `new_population`.
    * **Update:** Overwrite your current `population` with the `new_population`.
* **3. Return:** After the loop finishes (or breaks), return three things: the `best_history` list, the `avg_history` list, and the single best chromosome you found.


In [ ]:
# ENTER YOUR CODE HERE

### The Test Harness
*Let evolution begin! We will run your algorithm with a population of 100 for up to 250 generations.*


In [ ]:
# MILESTONE 3 TEST HARNESS

print("Initializing EvoTech Genetic Engine...")
best_hist, avg_hist, best_solution = run_genetic_algorithm(
    n_drones=N_DRONES,
    pop_size=100,
    max_generations=250,
    mutation_rate=0.05
)

# Print the final metrics and graph the learning curve
generations_run = len(best_hist)
final_fitness = calculate_fitness(best_solution)

print(f"\nEvolution Halted after {generations_run} generations.")
print(f"Best Configuration Found: {best_solution}")
print(f"Final Fitness Score: {final_fitness} / 28")

if final_fitness == get_max_fitness(N_DRONES):
    print("SUCCESS: A perfect defense grid was evolved!")
else:
    print("WARNING: The algorithm failed to find a perfect global optimum.")

# Print the board and plot the graph
print_board(best_solution)
plot_convergence(best_hist, avg_hist, N_DRONES)

### Architect's Audit (Milestone 3)

* **The 20-Drone Challenge:** Go back to the Setup Cell in Milestone 1 and change `N_DRONES = 8` to `N_DRONES = 20`. Re-run your cells. .
    1. **Scaling the Engine:** Using your original hyper-parameters (`pop_size=100`, `max_generations=250`, `mutation_rate=0.05`), what was the highest fitness your algorithm could reach on the 20x20 board? Why did the engine suddenly struggle?
    2. **Tuning for Complexity:** Tweak your hyper-parameters in the Test Harness to try and crack the 20x20 board. What specific values did you change (e.g., increasing population, increasing generations, altering mutation rate), and conceptually *why* did the larger problem space require those specific adjustments?

In [ ]:
# ENTER YOUR CODE HERE

ENTER YOUR ANSWERS HERE

## The Summary Audit
To complete the Lab, you must submit your final Colab Notebook containing your code along with brief 3-4 sentence answers to the following questions synthesizing the module:

1. **Evolutionary Architecture:** In your own words, what does a single "Chromosome" represent in this specific drones simulation? Conceptually, how does taking half of the DNA from one good parent and half from another theoretically lead to a better child?
2. **Convergence Dynamics:** Look at your generated Matplotlib graph for the 8-drone board. At roughly what generation did your population "converge," and how can you tell visually? If your Average Fitness line occasionally dipped down instead of strictly going up, which genetic operator caused those dips, and why is that healthy?
3. **Fitness Landscapes:** Based on Lecture 2.5, define a "Fitness Landscape." Given how easily your original algorithm got trapped when scaled up to the 20-drones board, would you describe its landscape as "Smooth" or "Rugged"? What do the "peaks" and "valleys" represent in this specific defense drone problem?
4. **Search-based AI vs. Bio-Inspired AI:** In Lab 1, you used Breadth-First Search (BFS) to find optimal paths. Why is it mathematically impossible to use Breadth-First Search to solve a massive 50x50 board? Explain why Evolutionary/Optimization algorithms are better suited for massive configuration problems like this.

ENTER YOUR ANSWERS HERE